# START HERE — AI Phishing Detection Project

هذا هو **دفتر التشغيل الرئيسي للمشروع كاملًا**. لا تحتاج إلى إعادة تشغيل `setup_env.bat`.

- إذا كانت البيئة `.venv` التي بدأت تثبيتها موجودة، سيستخدمها الدفتر تلقائيًا، وبالتالي **لن يضيع التثبيت الذي قمت به**.
- إذا كانت بعض مكتبات المرحلة الأساسية ناقصة، سيكمل الدفتر **الناقص فقط** بدون تنزيل PyTorch أو Transformers إلا إذا فعّلت تدريب DistilBERT بنفسك.
- المشروع يستخدم الداتا الصغيرة الموجودة داخل الحزمة فقط. **لن يتم تنزيل الـLarge Dataset**.
- الواجهة موجودة في `app.py` ويمكن تشغيلها من خلية داخل هذا الدفتر.

### المسار المقترح
ضع محتويات الحزمة في `D:\Ai projects` ثم افتح `START_HERE.ipynb` في VS Code واضغط **Run All**.


In [ ]:

# 1) Project engine — automatically reuses D:\Ai projects\.venv if it exists.
from pathlib import Path
import sys, subprocess, os, json, csv, socket, time, webbrowser

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    # Common case when VS Code starts the notebook from a different working directory.
    candidate = Path(r"D:\Ai projects")
    if candidate.exists():
        ROOT = candidate

VENV_PY = ROOT / ".venv" / "Scripts" / "python.exe"
EXEC_PY = VENV_PY if VENV_PY.exists() else Path(sys.executable)

print("Project root :", ROOT)
print("Execution Python:", EXEC_PY)
print("Reusing existing .venv:", VENV_PY.exists())

if not (ROOT / "src").exists():
    raise FileNotFoundError("Project folder not found. Open this notebook from H:\\assignment.")


In [ ]:

# 2) Core dependency check — installs only missing lightweight/core packages.
#    It deliberately does NOT install torch/transformers here.
core_imports = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "joblib": "joblib",
    "yaml": "pyyaml",
    "streamlit": "streamlit",
}

def import_ok(module):
    result = subprocess.run(
        [str(EXEC_PY), "-c", f"import {module}"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    return result.returncode == 0

missing = [pkg for module, pkg in core_imports.items() if not import_ok(module)]
if missing:
    print("Missing core packages:", missing)
    print("Installing only missing core requirements...")
    subprocess.check_call([str(EXEC_PY), "-m", "pip", "install", "-r", str(ROOT / "requirements_core.txt")])
else:
    print("Core environment is ready. Nothing to reinstall.")


In [ ]:

# 3) Validate the small dataset already included in the package.
processed = ROOT / "data" / "processed"
small_data = processed / "uploaded_unified_dedup.csv"
stats_file = processed / "dataset_stats.json"

print("Small dataset:", small_data)
print("Exists:", small_data.exists())
if stats_file.exists():
    stats = json.loads(stats_file.read_text(encoding="utf-8"))
    print(json.dumps(stats, indent=2, ensure_ascii=False))
else:
    print("dataset_stats.json not found, but the CSV can still be used.")

if not small_data.exists():
    raise FileNotFoundError("uploaded_unified_dedup.csv is missing from data/processed.")


In [ ]:

# 4) Build/refresh master_dataset.csv from the SMALL uploaded data only.
#    No Kaggle download is performed.
print("Preparing master dataset...")
subprocess.check_call([str(EXEC_PY), str(ROOT / "src" / "build_master_dataset.py")], cwd=str(ROOT))
print("Done:", ROOT / "data" / "processed" / "master_dataset.csv")


In [ ]:

# 5) Train the two baseline models and run cross-generator evaluation.
print("Training Logistic Regression and Linear SVM...")
subprocess.check_call([str(EXEC_PY), str(ROOT / "src" / "train_baselines.py")], cwd=str(ROOT))
print("Running cross-generator experiment...")
subprocess.check_call([str(EXEC_PY), str(ROOT / "src" / "evaluate_cross_generator.py")], cwd=str(ROOT))
print("Exporting visual-ready tables...")
subprocess.check_call([str(EXEC_PY), str(ROOT / "src" / "export_visual_ready.py")], cwd=str(ROOT))
print("Baseline stage complete.")


In [ ]:

# 6) Show the most important saved results directly in the notebook.
from IPython.display import display, HTML

def read_csv_rows(path, limit=20):
    path = Path(path)
    if not path.exists():
        return [], []
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        r = csv.reader(f)
        rows = list(r)
    if not rows:
        return [], []
    return rows[0], rows[1:limit+1]

def html_table(path, title):
    header, rows = read_csv_rows(path)
    if not header:
        display(HTML(f"<h3>{title}</h3><p>File not found or empty.</p>"))
        return
    parts=[f"<h3>{title}</h3><table style='border-collapse:collapse'>"]
    parts.append("<tr>"+"".join(f"<th style='border:1px solid #bbb;padding:6px'>{h}</th>" for h in header)+"</tr>")
    for row in rows:
        parts.append("<tr>"+"".join(f"<td style='border:1px solid #ddd;padding:6px'>{v}</td>" for v in row)+"</tr>")
    parts.append("</table>")
    display(HTML("".join(parts)))

html_table(ROOT / "results" / "baseline" / "baseline_metrics.csv", "Baseline Metrics")
html_table(ROOT / "results" / "baseline" / "cross_generator_phishing_detection.csv", "Cross-Generator Evaluation")


In [ ]:

# 7) Launch the Streamlit testing interface directly from the notebook.
#    The cell is non-blocking; the notebook remains usable.
def port_open(host="127.0.0.1", port=8501):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.4)
        return s.connect_ex((host, port)) == 0

url = "http://localhost:8501"
if port_open():
    print("Interface is already running:", url)
else:
    log_path = ROOT / "results" / "streamlit.log"
    log_file = open(log_path, "a", encoding="utf-8")
    proc = subprocess.Popen(
        [str(EXEC_PY), "-m", "streamlit", "run", str(ROOT / "app.py"),
         "--server.headless=true", "--server.port=8501"],
        cwd=str(ROOT),
        stdout=log_file,
        stderr=subprocess.STDOUT,
        creationflags=getattr(subprocess, "CREATE_NEW_PROCESS_GROUP", 0),
    )
    (ROOT / "results" / "streamlit.pid").write_text(str(proc.pid), encoding="utf-8")
    time.sleep(4)
    print("Interface started:", url)
    print("PID:", proc.pid)
    print("Log:", log_path)

try:
    webbrowser.open(url)
except Exception:
    pass
print("Open in browser if it did not open automatically:", url)


## Optional: DistilBERT

المشروع كامل ويحتوي كود DistilBERT أيضًا، لكن **لن يبدأ التدريب تلقائيًا** حتى لا يستهلك وقتك أو يعيد تنزيل PyTorch بدون رغبتك.

إذا أردت تدريب DistilBERT لاحقًا، غيّر `TRAIN_DISTILBERT = False` إلى `True` في الخلية التالية ثم شغلها. بعد نجاح التدريب سيظهر **DistilBERT تلقائيًا داخل نفس الواجهة**.


In [ ]:

# 8) OPTIONAL transformer stage. Leave False for the current small-data baseline phase.
TRAIN_DISTILBERT = False

if TRAIN_DISTILBERT:
    optional_imports = ["torch", "transformers", "datasets", "accelerate", "scipy"]
    missing_optional = [m for m in optional_imports if not import_ok(m)]
    if missing_optional:
        print("Installing optional transformer requirements:", missing_optional)
        subprocess.check_call([str(EXEC_PY), "-m", "pip", "install", "-r", str(ROOT / "requirements_transformer.txt")])
    print("Training DistilBERT. This can take significant time on CPU...")
    subprocess.check_call([
        str(EXEC_PY), str(ROOT / "src" / "train_distilbert.py"),
        "--epochs", "2", "--batch-size", "8", "--max-length", "256"
    ], cwd=str(ROOT))
    print("DistilBERT saved to:", ROOT / "results" / "distilbert_model")
    print("Restart Streamlit after training so the new model appears in the model selector.")
else:
    print("DistilBERT training skipped (TRAIN_DISTILBERT=False). Baseline project remains fully runnable.")


In [ ]:

# 9) Project output map — useful when preparing your visuals/report.
paths = [
    ROOT / "data" / "processed" / "master_dataset.csv",
    ROOT / "results" / "baseline" / "baseline_metrics.csv",
    ROOT / "results" / "baseline" / "baseline_confusion_matrices.csv",
    ROOT / "results" / "baseline" / "cross_generator_phishing_detection.csv",
    ROOT / "results" / "visual_ready",
    ROOT / "app.py",
]
for p in paths:
    print(("[OK] " if p.exists() else "[--] ") + str(p))

print("\nWorkflow complete. Use the Streamlit interface at http://localhost:8501")
